### Create Polygon

In [1]:
# import rasterio
# from rasterio.features import shapes
# import numpy as np
# from shapely.geometry import shape, mapping
# import fiona
# from scipy.ndimage import label
# import matplotlib.pyplot as plt
# import reverse_geocoder as rg
# import pycountry
# from rasterio.plot import show
# import os
# from pyproj import Transformer, CRS

# # Step 1: Read the raster file
# input_raster_path = r'D:\RIDA\polygontest\from pppo\output_band_1.tif'
# output_shapefile_path = r'D:\RIDA\polygontest\Output\burn_condition.shp'

# with rasterio.open(input_raster_path) as src:
#     raster_data = src.read(1)  # Read the first band
#     transform = src.transform  # Get the affine transform
#     crs = src.crs  # Get the CRS of the input raster

# # Create a transformer to convert from the raster's CRS to WGS84
# transformer = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)

# # Step 2: Extract features based on burn condition values
# burn_condition = (raster_data == 1).astype(np.uint8)

# # Label connected components
# labeled_array, num_features = label(burn_condition)

# # Step 3: Convert labeled features to polygons
# shapes_generator = shapes(labeled_array, transform=transform)

# # Create a figure and axis
# fig, ax = plt.subplots(figsize=(12, 10))

# # Plot the raster
# with rasterio.open(input_raster_path) as src:
#     show(src, ax=ax, cmap='viridis', title="Burn Condition Raster with Polygons")

# plt.show()

# polygons = []
# for geom, value in shapes_generator:
#     if value > 0:  # Only take the features corresponding to burn condition
#         polygons.append(shape(geom))

# # Use a projected CRS for area calculation (UTM zone 47N)
# projected_crs = CRS.from_epsg(32647)
# transformer_to_utm = Transformer.from_crs(crs, projected_crs, always_xy=True)

# projected_polygons = []
# for polygon in polygons:
#     # Reproject polygon to UTM
#     projected_polygon = shape(mapping(polygon))
#     projected_polygon = shape({
#         'type': 'Polygon',
#         'coordinates': [
#             [
#                 transformer_to_utm.transform(x, y) for x, y in polygon.exterior.coords
#             ]
#         ]
#     })
#     projected_polygons.append(projected_polygon)

# # Calculate total area and save shapefile
# total_area = sum(polygon.area for polygon in projected_polygons)

# schema = {
#     'geometry': 'Polygon',
#     'properties': {'id': 'int', 'area_m2': 'float'},
# }

# os.makedirs(os.path.dirname(output_shapefile_path), exist_ok=True)

# with fiona.open(output_shapefile_path, 'w', 'ESRI Shapefile', schema=schema, crs=crs) as shp:
#     for i, polygon in enumerate(projected_polygons):
#         area_m2 = polygon.area
#         shp.write({
#             'geometry': mapping(polygon),
#             'properties': {'id': i + 1, 'area_m2': area_m2},
#         })

# print(f"Shapefile saved to {output_shapefile_path}")
# print(f"Total burn area: {total_area:.2f} square meters\n")

# # Reverse geocode to get city and province
# def get_location_info(x, y):
#     lon, lat = transformer.transform(x, y)
#     coordinates = (lat, lon)  # Note the order: (latitude, longitude)
#     result = rg.search(coordinates)
#     if result:
#         location = result[0]
#         city = location['name']
#         province = location['admin1']
#         country_code = location['cc']
#         try:
#             country = pycountry.countries.get(alpha_2=country_code).name
#         except AttributeError:
#             country = "Unknown"
#         return city, province, country, lat, lon
#     return None, None, None, lat, lon

# # Step 5: Print properties and plot the output
# fig, ax = plt.subplots(figsize=(10, 10))

# # Plot the original raster data
# ax.imshow(raster_data, cmap='gray', extent=(transform[2], transform[2] + transform[0] * raster_data.shape[1], 
#                                             transform[5] + transform[4] * raster_data.shape[0], transform[5]))
# ax.set_title("Burn Condition Raster and Polygons")

# # Plot the polygons and print their properties
# for i, polygon in enumerate(polygons):
#     x, y = polygon.exterior.xy
#     ax.plot(x, y, color='red', linewidth=2)
    
#     # Get centroid of the polygon for reverse geocoding
#     centroid = polygon.centroid
#     city, province, country, lat, lon = get_location_info(centroid.x, centroid.y)
#     area_m2 = projected_polygons[i].area  # Get area from projected polygon
    
#     # Annotate the polygon with its ID
#     ax.annotate(str(i+1), (centroid.x, centroid.y), color='white', fontweight='bold', ha='center', va='center')
    
#     print(f"Polygon {i+1}:")
#     print(f"  Centroid: ({lat:.6f}, {lon:.6f})")
#     print(f"  City: {city}, Province: {province}, Country: {country}")
#     print(f"  Area: {area_m2:.4f} square meters")
#     print()

# plt.tight_layout()
# plt.show()


import rasterio
from rasterio.features import shapes
import numpy as np
from shapely.geometry import shape, mapping
import fiona
from scipy.ndimage import label
import reverse_geocoder as rg
import pycountry
from pyproj import Transformer, CRS
import os

# Step 1: Read the raster file
input_raster_path = r'D:\RIDA\polygontest\from pppo\T47QNB_20200330T034529_combined.tif'
output_shapefile_path = r'D:\RIDA\polygontest\Output\burn_condition.shp'

with rasterio.open(input_raster_path) as src:
    raster_data = src.read(1)  # Read the first band
    transform = src.transform  # Get the affine transform
    crs = src.crs  # Get the CRS of the input raster

# Create a transformer to convert from the raster's CRS to WGS84
transformer = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)

# Step 2: Extract features based on burn condition values
burn_condition = (raster_data == 1).astype(np.uint8)

# Label connected components
labeled_array, num_features = label(burn_condition)

# Step 3: Convert labeled features to polygons
shapes_generator = shapes(labeled_array, transform=transform)

polygons = []
for geom, value in shapes_generator:
    if value > 0:  # Only take the features corresponding to burn condition
        polygons.append(shape(geom))

# Use a projected CRS for area calculation (UTM zone 47N)
projected_crs = CRS.from_epsg(32647)
transformer_to_utm = Transformer.from_crs(crs, projected_crs, always_xy=True)

projected_polygons = []
for polygon in polygons:
    # Reproject polygon to UTM
    projected_polygon = shape(mapping(polygon))
    projected_polygon = shape({
        'type': 'Polygon',
        'coordinates': [
            [
                transformer_to_utm.transform(x, y) for x, y in polygon.exterior.coords
            ]
        ]
    })
    projected_polygons.append(projected_polygon)

# Calculate total area and save shapefile
total_area = sum(polygon.area for polygon in projected_polygons)

schema = {
    'geometry': 'Polygon',
    'properties': {'id': 'int', 'area_m2': 'float'},
}

os.makedirs(os.path.dirname(output_shapefile_path), exist_ok=True)

with fiona.open(output_shapefile_path, 'w', 'ESRI Shapefile', schema=schema, crs=crs) as shp:
    for i, polygon in enumerate(projected_polygons):
        area_m2 = polygon.area
        shp.write({
            'geometry': mapping(polygon),
            'properties': {'id': i + 1, 'area_m2': area_m2},
        })

print(f"Shapefile saved to {output_shapefile_path}")
print(f"Total burn area: {total_area:.2f} square meters\n")

# Reverse geocode to get city and province
def get_location_info(x, y):
    lon, lat = transformer.transform(x, y)
    coordinates = (lat, lon)  # Note the order: (latitude, longitude)
    result = rg.search(coordinates)
    if result:
        location = result[0]
        city = location['name']
        province = location['admin1']
        country_code = location['cc']
        try:
            country = pycountry.countries.get(alpha_2=country_code).name
        except AttributeError:
            country = "Unknown"
        return city, province, country, lat, lon
    return None, None, None, lat, lon

# Step 5: Print reverse geocoding information
for i, polygon in enumerate(polygons):
    # Get centroid of the polygon for reverse geocoding
    centroid = polygon.centroid
    city, province, country, lat, lon = get_location_info(centroid.x, centroid.y)
    
    print(f"Polygon {i+1}:")
    print(f"  Centroid: ({lat:.6f}, {lon:.6f})")
    print(f"  City: {city}, Province: {province}, Country: {country}")
    print()



Shapefile saved to D:\RIDA\polygontest\Output\burn_condition.shp
Total burn area: 5436700.00 square meters

Loading formatted geocoded file...
Polygon 1:
  Centroid: (19.895818, 99.369288)
  City: Mae Ai, Province: Chiang Mai, Country: Thailand

Polygon 2:
  Centroid: (19.895827, 99.382261)
  City: Mae Ai, Province: Chiang Mai, Country: Thailand

Polygon 3:
  Centroid: (19.895728, 99.369097)
  City: Mae Ai, Province: Chiang Mai, Country: Thailand

Polygon 4:
  Centroid: (19.895519, 99.459088)
  City: Mae Ai, Province: Chiang Mai, Country: Thailand

Polygon 5:
  Centroid: (19.895569, 99.358683)
  City: Fang, Province: Chiang Mai, Country: Thailand

Polygon 6:
  Centroid: (19.895546, 99.369574)
  City: Mae Ai, Province: Chiang Mai, Country: Thailand

Polygon 7:
  Centroid: (19.895457, 99.369287)
  City: Mae Ai, Province: Chiang Mai, Country: Thailand

Polygon 8:
  Centroid: (19.895388, 99.358826)
  City: Fang, Province: Chiang Mai, Country: Thailand

Polygon 9:
  Centroid: (19.895366, 99

KeyboardInterrupt: 

In [ ]:
# from pyproj import Transformer


# # Assuming the current coordinates are in some UTM projection
# # You'll need to replace EPSG:32601 with the correct UTM zone for Myanmar
# transformer = Transformer.from_crs("epsg:32647", "EPSG:4326", always_xy=True)

# x, y = 2409156.6666666665, 479225.0  # Example coordinates from your output
# lon, lat = transformer.transform(x, y)

# print(f"Latitude: {lat}, Longitude: {lon}")

In [ ]:
# import rasterio
# from rasterio.plot import show

# # Path to your TIFF file
# # tiff_path = 'D:\RIDA\polygontest\Read Band\T47QNB_20210325T034539_combined.tif'
# # tiff_path = 'D:\RIDA\polygontest\output_band_1.tif'

# # Open the TIFF file
# with rasterio.open(tiff_path) as src:
#     # Print metadata
#     print("Metadata:")
#     print(src.meta)

#     # Get the number of bands
#     num_bands = src.count
#     print(f"The file has {num_bands} bands.")

#     # Read and print each band
#     for i in range(1, num_bands + 1):
#         band = src.read(i)
#         print(f"Band {i} data:")
#         print(band)

#         # Optionally, display each band
#         show(band, cmap='gray', title=f'Band {i}')

### CSV ADD TO TIF

In [ ]:
# import rasterio
# import numpy as np
# import pandas as pd
# from rasterio.transform import from_origin

# # Step 1: Read the original raster file to get its metadata
# input_raster_path = 'path_to_your_input_raster.tif' #tif prepare from dataen
# with rasterio.open(input_raster_path) as src:
#     transform = src.transform
#     crs = src.crs
#     width = src.width
#     height = src.height
#     dtype = src.dtypes[0]

# # Step 2: Read the CSV file containing the predictions
# predictions_csv_path = 'path_to_your_predictions.csv'
# df = pd.read_csv(predictions_csv_path)

# # Assume the CSV has columns 'row', 'col', and 'prediction'
# # You may need to adjust the column names based on your actual CSV format
# rows = df['row'].values
# cols = df['col'].values
# predictions = df['prediction'].values

# # Step 3: Create an empty array to hold the predictions
# prediction_array = np.zeros((height, width), dtype=dtype)

# # Step 4: Fill the array with the predictions
# for row, col, prediction in zip(rows, cols, predictions):
#     prediction_array[row, col] = prediction

# # Step 5: Save the prediction array as a new GeoTIFF file
# output_raster_path = 'path_to_your_output_raster.tif' #tif label from ML

# with rasterio.open(output_raster_path, 'w', driver='GTiff', height=height, width=width, count=1, dtype=dtype, crs=crs, transform=transform) as dst:
#     dst.write(prediction_array, 1)

# print(f"Prediction raster saved to {output_raster_path}")


### Download

In [ ]:
# """
#  Objective : This Script for Download Satellite data : Sentinel-2 from Copernicus Data Space Ecosystem
#              referense https://dataspace.copernicus.eu/
#              This Script is
#                + Can be to autoamtic download by using Task scheduler on windows
#                + will check with data to download already (not download)
#                + will check on file to downlaod already but not compleate (failed file) and will redownload
#                + can use many users and random user for protect limited from server service
#                + can select image of Sentinel-2 by Tiles easy than by coordinate
# Writer : Anusorn Rungsipanich : Gistda@2023, RESGAT@2023

# log status
# 20231106 : Starting write script
# 20231113 : frist version 


# """
# # import session
# import datetime,os,subprocess,glob,csv,random,time
# from datetime import timedelta
# from datetime import date
# import requests
# import pandas as pd

# #  Decare Session
# # --- Decare log Floder ---
# PathLog = 'D:/RIDA/satallite_image_acquire/log'  # Floder for keep log file


# # --- Decare Option ---
# DateOption = 1 # 1 = Number of from now, 2 = Start Day to End Day
# NuDays = 5 # nuber of day from now with DateOption = 1
# StartDay = '2022-11-06' # YYYYMMDD  Starting Date with DateOption = 2
# EndDay = '2023-01-12' # YYYYMMDD Ending Date with DateOption = 2
# SepDays = 10 # for separate day every SepDays No need to change 10 days is work

# # --- Decare User Password ---

# NuUser = 3 # Nuber os user
# User01 = 'Desukaz.skrr@gmail.com' #user name 1
# Pass01 = 'IMDSK7901Desuka!' #password user 1
# User02 = 'Desukaz.skrr@gmail.com' #user name 2
# Pass02 = 'IMDSK7901Desuka!' #password user 2
# User03 = 'Desukaz.skrr@gmail.com' #user name 3
# Pass03 = 'IMDSK7901Desuka!' #password user 3


# MainSN = 'D:/RIDA/satallite_image_acquire'   # Floder for keeping satellite data 

# Sn2Levels = ['MSIL1C'] #['MSIL1C','MSIL2A'] option of level to download

# aoi = "POLYGON((96.7 21.0,106.0 21.0,106.0 5.2,96.7 5.2,96.7 21.0))'" #Thailand rectangular boundary
# data_collection = "SENTINEL-2" #SENTINEL-1,SENTINEL-2,SENTINEL-3,SENTINEL-5P

# Tiles = ['T47PRQ', 'T47PRR', 'T48PTA', 'T48PTV', 'T47PNQ', 'T47PNR', 'T47PNS', 'T47PPQ'] # Tile for download

# #Perpare session
# lstUser = []
# lstPass = []
# for u in range(NuUser):
#     lstUser.append(globals()['User' +  str(u + 1).zfill(2)])
#     lstPass.append(globals()['Pass' +  str(u + 1).zfill(2)])
                 

# #Module session
# def DateDolist():
#     TDay = datetime.date.today()
#     SYMD = TDay - timedelta(NuDays)
#     EnD = TDay.strftime("%Y-%m-%d")
#     StD = SYMD.strftime("%Y-%m-%d")
#     return [StD,EnD]

# def Sepby10day(lSE):
#     lstSepSE = []
#     for SE in lSE:
#         FristStart = datetime.datetime.strptime(SE[0], '%Y-%m-%d')
#         FristEnd = datetime.datetime.strptime(SE[1], '%Y-%m-%d') + datetime.timedelta(days = 1)
#         #DeltaDay = FristEnd - FristStart
#         #div = DeltaDay / datetime.timedelta(11)
#         RunStartDay = FristStart
#         while RunStartDay < FristEnd:
#             RunEndDay = RunStartDay + datetime.timedelta(days = 10)
#             if RunEndDay > FristEnd:
#                 RunEndDay = FristEnd
#             lstSepSE.append([RunStartDay.strftime("%Y-%m-%d"),RunEndDay.strftime("%Y-%m-%d")])
            
#             #print (RunStartDay,RunEndDay)
#             RunStartDay = RunStartDay + datetime.timedelta(days = 10)
#     return (lstSepSE)
        

# def CalStrEnd():
#     if DateOption == 1:
#         DoDay = DateDolist()
#         SDay = DoDay[0]
#         EDay  = DoDay[1]
#     elif DateOption == 2:
#         SDay = StartDay # YYYY-MM-DD
#         EDay = EndDay

#     if SDay[:4] == EDay[:4]:
#         listStrEnd = [[SDay,EDay]]
#     else:
#         listStrEnd = [[SDay,SDay[:5]+'12-31'],[EDay[:5]+'01-01',EDay]]
#     lstSepStEn = Sepby10day(listStrEnd)
#     return lstSepStEn

# def get_keycloak(username: str, password: str) -> str:
#     data = {"client_id": "cdse-public","username": username,"password": password,"grant_type": "password",}
#     try:
#         r = requests.post("https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token",data=data,)
#         r.raise_for_status()
#     except Exception as e:
#         raise Exception(f"Keycloak token creation failed. Reponse from the server was: {r.json()}")
#     return r.json()["access_token"]
        


# def SearchIdSN(SE,Tile):
#     IdNameSen = []
#     start_date = SE[0]
#     end_date = SE[1]
    
#     #json = requests.get(f"https://catalogue.dataspace.copernicus.eu/odata/v1/Products?$filter=Collection/Name eq '{data_collection}' and OData.CSC.Intersects(area=geography'SRID=4326;{aoi}) and ContentDate/Start gt {start_date}T00:00:00.000Z and ContentDate/Start lt {end_date}T00:00:00.000Z").json()
#     #json = requests.get(f"https://catalogue.dataspace.copernicus.eu/odata/v1/Products?$filter=contains(Name,'T47PRS') and Collection/Name eq '{data_collection}' and OData.CSC.Intersects(area=geography'SRID=4326;{aoi}) and ContentDate/Start gt {start_date}T00:00:00.000Z and ContentDate/Start lt {end_date}T00:00:00.000Z").json()
#     json = requests.get(f"https://catalogue.dataspace.copernicus.eu/odata/v1/Products?$filter=contains(Name,'{Tile}') and Collection/Name eq '{data_collection}' and OData.CSC.Intersects(area=geography'SRID=4326;{aoi}) and ContentDate/Start gt {start_date}T00:00:00.000Z and ContentDate/Start lt {end_date}T00:00:00.000Z").json()

#     AllSn = pd.DataFrame.from_dict(json['value']).head(20)
#     #my_list = list(AllSn)
#     #print (my_list)
    
#     IdNameSn = pd.DataFrame(AllSn, columns=['Id', 'Name','Checksum','ContentLength'])
#     for index,row in IdNameSn.iterrows():
#         IdNameSen.append ([row['Id'],row['Name'],row['Checksum'],row['ContentLength']])
#     return IdNameSen


# def CheckSenLevel(SEDay,Prod):
#     PdFil = []
#     if len(Prod):
#         for PdT in Prod:
#             NameProd = PdT[1][:-5] + '.zip'
#             #print (NameProd)
#             if (NameProd[4:10] in Sn2Levels):
#                 PdFil.append(PdT)
#     return PdFil

# def CheckSenArch(SEDay,Prod):
#     PdFil = []
#     YY = SEDay[0][:4]
#     ActiveDir = MainSN + "/" + YY
#     if not os.path.exists(ActiveDir):
#         os.makedirs(ActiveDir)    
#     os.chdir(ActiveDir)
#     AcrNameList = glob.glob("*.zip")
#     #print (AcrNameList)
#     #AcrMainNameList = []
#     if len(Prod):
#         for PdT in Prod:
#             NameProd = PdT[1][:-5] + '.zip'
#             #print (NameProd,(NameProd in AcrNameList))
#             if (NameProd in AcrNameList):
#                 sizefile = os.path.getsize(NameProd)
#                 #print (PdT[2][0]['Value'] , arhFile,os.path.getsize(NameProd),PdT[3])
#                 #if not (PdT[2][0]['Value'] == arhFile):
#                 if sizefile < PdT[3]:
#                     os.remove(NameProd)
#                     PdFil.append(PdT)
#                     print("        * File size is smaller than original then Will Remove and reDownload : " + NameProd)
#                     logFile.write("        * File size is smaller than original then Will Remove and reDownload : " + NameProd + "\n")
                    
#             else:
#                 PdFil.append(PdT)
#                 print("        * Will Download : " + NameProd)
#                 logFile.write("        * Will Download : " + NameProd + "\n")
                
#     return PdFil

# def DownSen(SEDay,Prod):
#     YY = SEDay[0][:4]
#     ActiveDir = MainSN + "/" + YY
#     os.chdir(ActiveDir)
#     if len(Prod):
#         for PdT in Prod:
#             #for random change user
#             randUser = random.randrange(NuUser)
#             UserN = lstUser[randUser]
#             PassW = lstPass[randUser]
            
#             keycloak_token = get_keycloak(UserN, PassW)
#             session = requests.Session()
#             session.headers.update({'Authorization': f'Bearer {keycloak_token}'})
#             print ("          + Using User : " + UserN + " to download.")
#             logFile.write("          + Using User : " + UserN + " to download.\n")

#             url = f'https://catalogue.dataspace.copernicus.eu/odata/v1/Products(' + PdT[0] + f')/$value'
#             #print (url)
#             NameProd = PdT[1][:-5] + '.zip'
#             print ("          + Starting Download : " + str(datetime.datetime.now()) + " : " + NameProd)
#             logFile.write("          + Starting Download : " + str(datetime.datetime.now()) + " : " + NameProd + "\n")
#             response = session.get(url, allow_redirects=False)
#             while response.status_code in (301, 302, 303, 307):
#                 url = response.headers['Location']
#                 response = session.get(url, allow_redirects=False)
#             file = session.get(url, verify=False, allow_redirects=True)
#             with open(NameProd, 'wb') as p:
#                 p.write(file.content)
#             print ("          + Ending Download : " + str(datetime.datetime.now()) + " : " + NameProd)
#             logFile.write("          + Ending Download : " + str(datetime.datetime.now()) + " : " + NameProd + "\n")


# if __name__ == "__main__":
#     logDate = datetime.datetime.now()
#     logName = "DownSN_GISTDA_LOG_" + str(logDate.year) + str(logDate.month).zfill(2) + str(logDate.day).zfill(2) + str(logDate.hour).zfill(2) + str(logDate.minute).zfill(2) + ".txt"
#     logFile = open(PathLog + "/" + logName, 'w')
#     print ("  Download Sentinal-2 file")
#     logFile.write('  Download Sentinal-2 file' + '\n')
#     print ('  Script Download Sentinal-2 From Gistda/RESGAT Version 1.10')
#     logFile.write('  Script Download Sentinal-2 From Gistda/RESGAT Version 1.10\n')
#     #logFile.write("Process file since : ", date.today() - timedelta(NuDays) ," until ",date.today())
#     print ("  Starting time is : " + str(datetime.datetime.now()))
#     logFile.write("  Starting time is : " + str(datetime.datetime.now()) + "\n")

#     StrEnd = CalStrEnd()
#     #print (StrEnd[0][0])
#     #for random change user
#     randUser = random.randrange(NuUser)
#     UserN = lstUser[randUser]
#     PassW = lstPass[randUser]
#     keycloak_token = get_keycloak(UserN, PassW)

#     for SE in StrEnd:
#         print ("      -   Search from " + SE[0] + " to " + SE[1])
#         logFile.write("      -   Search from " + SE[0] + " to " + SE[1] + "\n")
#         #Search file
#         for Tile in Tiles:
#             SN2Pd = SearchIdSN(SE,Tile)
#             #Filter only level
#             FilSN2Lev = CheckSenLevel(SE,SN2Pd)
#             #print (FilSN2Lev)
#             #filter if have in 
#             FilSN2Pd = CheckSenArch(SE,FilSN2Lev)
#             #print (FilSN2Pd)
#             DownSen(SE,FilSN2Pd)

#     #Round 2
#     StrEnd = CalStrEnd()
#     #print (StrEnd[0][0])
#     #for random change user
#     randUser = random.randrange(NuUser)
#     UserN = lstUser[randUser]
#     PassW = lstPass[randUser]
#     keycloak_token = get_keycloak(UserN, PassW)

#     for SE in StrEnd:
#         print ("      -   Search from " + SE[0] + " to " + SE[1])
#         logFile.write("      -   Search from " + SE[0] + " to " + SE[1] + "\n")
#         #Search file
#         for Tile in Tiles:
#             SN2Pd = SearchIdSN(SE,Tile)
#             #Filter only level                                                                          
#             FilSN2Lev = CheckSenLevel(SE,SN2Pd)
#             #print (FilSN2Lev)
#             #filter if have in 
#             FilSN2Pd = CheckSenArch(SE,FilSN2Lev)
#             #print (FilSN2Pd)
#             DownSen(SE,FilSN2Pd)


#     print ("  DONE Download Sentinal-2 !!!")
#     logFile.write("  DONE Download Sentinal-2 !!!\n")
#     print ("  Ending time is : " + str(datetime.datetime.now()))
#     logFile.write("  Ending time is : " + str(datetime.datetime.now()) + "\n")

#     logFile.close()        
    

### Sentinel

In [ ]:
# SentinelProcess_Train_rNDI.py

import os
import shutil
import time
from threading import Timer
from time import gmtime, strftime
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from skimage import measure
import rasterio
from rasterio.transform import from_origin
from rasterio.enums import Resampling
import geopandas as gpd
from rasterio.features import shapes
from shapely.geometry import shape
from xml.etree import ElementTree as ET
import math
from rasterio.warp import transform
from osgeo import gdal
import warnings
import csv
import json
warnings.filterwarnings(action='ignore')

# Load the configuration values
with open("D:\RIDA\config_rndi.json", "r") as f:
    config = json.load(f)

# Extracting configuration values
burn_con_lv1_threshold_rndi = config["burn_con_lv1_threshold_rndi"]
burn_con_lv3_threshold_rndi = config["burn_con_lv3_threshold_rndi"]
burn_con_lv4_threshold_rndi = config["burn_con_lv4_threshold_rndi"]

systemCooldown = 2
Error_Limit = 1
mode = True

# Define paths with double backslashes
Drive = "D:\RIDA\sentinel_process"
Image = os.path.join(Drive, "Image")
Image_Pre = os.path.join(Drive, "Image_Pre")
Image_Finish = os.path.join(Drive, "Image_Finish")
Image_Missing = os.path.join(Drive, "Image_Missing")
Output = os.path.join(Drive, "Output")
Rtbcon = os.path.join(Drive, "Raster_BurnCon")
Rtbreg = os.path.join(Drive, "Raster_BurnReg")
RtbShape = os.path.join(Drive, "Raster_BurnShape")
RtbLevel = os.path.join(Drive, "Raster_BurnLevel")

# Track_arr = [
#     "Area1\\", "Area2\\", "Area3\\", "Area4\\", "Area5\\", "Area6\\", "Area7\\", "Area8\\", "Area9\\", "Area10\\", "Area11\\", "Area12\\",
#     "Area00\\", "Area01\\", "Area02\\", "Area03\\", "Area04\\", "Area05\\", "Area06\\", "Area07\\", "Area08\\", "Area09\\",
#     "Area10\\", "Area11\\", "Area12\\", "Area13\\", "Area14\\", "Area15\\", "Area16\\", "Area17\\", "Area18\\", "Area19\\",
#     "Area20\\", "Area21\\", "Area22\\", "Area23\\", "Area24\\", "Area25\\", "Area26\\", "Area27\\", "Area28\\", "Area29\\",
#     "Area30\\", "Area31\\", "Area32\\", "Area33\\", "Area34\\", "Area35\\", "Area36\\", "Area37\\", "Area38\\", "Area39\\",
#     "Area40\\", "Area41\\", "Area42\\", "Area43\\", "Area44\\", "Area45\\", "Area46\\", "Area47\\", "Area48\\", "Area49\\",
#     "Area50\\", "Area51\\", "Area52\\", "Area53\\", "Area54\\", "Area55\\", "Area56\\", "Area57\\", "Area58\\", "Area59\\",
#     "Area60\\", "Area61\\", "Area62\\", "Area63\\", "Area64\\", "Area65\\", "Area66\\", "Area67\\", "Area68\\", "Area69\\",
#     "Area70\\", "Area71\\", "Area72\\", "Area73\\", "Area74\\", "Area75\\", "Area76\\", "Area77\\", "Area78\\", "Area79\\",
#     "Area80\\", "Area81\\", "Area82\\", "Area83\\", "Area84\\", "Area85\\"
# ]

Track_arr = ["T47QME\\"]

def loadCooldown():
    global mode

def print_time():
    return time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

def save_as_geotiff(data, output_path, transform, crs):
    height, width = data.shape
    profile = {
        'driver': 'GTiff',
        'count': 1,
        'dtype': 'uint8',
        'width': width,
        'height': height,
        'crs': crs,
        'transform': transform,
        'compress': 'packbits',
        'tiled': True,
        'interleave': 'band',
    }

    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(data.astype('uint8'), 1)
        
        # Add GeoTransform metadata
        dst.update_tags(AREA_OR_POINT='Area')
        
        # Calculate and add corner coordinates
        left, top = transform * (0, 0)
        right, bottom = transform * (width, height)
        dst.update_tags(CORNER_UL_LON_LAT=f"{left}, {top}")
        dst.update_tags(CORNER_UR_LON_LAT=f"{right}, {top}")
        dst.update_tags(CORNER_LL_LON_LAT=f"{left}, {bottom}")
        dst.update_tags(CORNER_LR_LON_LAT=f"{right}, {bottom}")


        
def burn_con(rNDI, dNDVI, data_AFB03, data_AFB08, data_AFB8A10, data_AFB02, data_AFB04, data_AFB0510, data_AFB0610, data_AFB0710, data_AFB0910, data_AFB1210):
    burn_con_lv1 = np.where((rNDI > burn_con_lv1_threshold_rndi), 1, 0)
    burn_con_lv2 = np.where((data_AFB08 > data_AFB03), 1, 0)
    burn_con_lv3 = np.where((dNDVI < burn_con_lv3_threshold_rndi), 1, 0)
    burn_con_lv4 = np.where((data_AFB08 < burn_con_lv4_threshold_rndi), 1, 0)
    burnCon_Final = np.where(((burn_con_lv1 == 1) & (burn_con_lv2 == 1) & (burn_con_lv3 == 1) & (burn_con_lv4 == 1)), 1, 0)
    burnCon_Final = np.where(
        np.any(
            (
                (data_AFB08 < 100), (data_AFB8A10 < 100), (data_AFB03 < 100),
                (data_AFB02 < 100), (data_AFB04 < 100), (data_AFB0510 < 100),
                (data_AFB0610 < 100), (data_AFB0710 < 100), (data_AFB0910 < 100),
                (data_AFB1210 < 100),
            ),
            axis=0,
        ),
        0,
        burnCon_Final,
    )
    return burnCon_Final

def burn_region(data_BCON, threshold_value):
    burnRegionGrp = measure.label(data_BCON, connectivity=1)
    print(print_time() + f"Raster_Process :: Region > {threshold_value}")

    burnRegionGrpThresholded = np.where(burnRegionGrp > threshold_value, 1, 0)
    burnRegion = np.where(burnRegionGrpThresholded == 1, 1, 0)

    print(print_time() + "Raster_Process :: Save Raster")
    
    return burnRegion


def Move_File(FileName, CurrDir, DestDir):
    try:
        if os.path.exists(os.path.join(CurrDir, FileName)):
            if os.path.exists(os.path.join(DestDir, FileName)):
                os.remove(os.path.join(DestDir, FileName))
            shutil.copy(os.path.join(CurrDir, FileName), DestDir)
            t = Timer(1, loadCooldown)
            t.start()
            t.join()
            os.remove(os.path.join(CurrDir, FileName))
            print(print_time() + f"Raster_Process :: Move File {FileName} Complete")
    except Exception as e:
        print(print_time() + f"Raster_Process :: Can not Move File {FileName}")
        print(print_time() + str(e))

def Raster_Process(Track):
    global mode, Image, Image_Pre, Image_Finish, Image_Missing, Output, Error_Limit
    Loop_Limit = 0
    image_track = os.path.join(Image, Track)
    rasters = [r for r in os.listdir(image_track) if r.endswith('B1210.jp2')]

    for raster in rasters:
        if Loop_Limit > 0:
            mode = True
            return
        Full_name = os.path.splitext(raster)[0]
        Mid_name = Full_name[:23]
        Short_name = Full_name[:6]

        BFB02 = os.path.join(Image_Pre, Track, f"{Short_name}_B02.jp2")
        BFB03 = os.path.join(Image_Pre, Track, f"{Short_name}_B03.jp2")
        BFB04 = os.path.join(Image_Pre, Track, f"{Short_name}_B04.jp2")
        BFB08 = os.path.join(Image_Pre, Track, f"{Short_name}_B08.jp2")
        BFB1210 = os.path.join(Image_Pre, Track, f"{Short_name}_B1210.jp2")

        AFB02 = os.path.join(Image, Track, f"{Mid_name}B02.jp2")
        AFB03 = os.path.join(Image, Track, f"{Mid_name}B03.jp2")
        AFB04 = os.path.join(Image, Track, f"{Mid_name}B04.jp2")
        AFB0510 = os.path.join(Image, Track, f"{Mid_name}B0510.jp2")
        AFB0610 = os.path.join(Image, Track, f"{Mid_name}B0610.jp2")
        AFB0710 = os.path.join(Image, Track, f"{Mid_name}B0710.jp2")
        AFB08 = os.path.join(Image, Track, f"{Mid_name}B08.jp2")
        AFB8A10 = os.path.join(Image, Track, f"{Mid_name}B8A10.jp2")
        AFB0910 = os.path.join(Image, Track, f"{Mid_name}B0910.jp2")
        AFB1210 = os.path.join(Image, Track, f"{Mid_name}B1210.jp2")

        tile = Short_name
        date = Full_name[7:15] 
        
        print(print_time() + f"Checking files for {Full_name}")
        file_check = {
            "BFB08": os.path.exists(BFB08),
            "BFB1210": os.path.exists(BFB1210),
            "AFB02": os.path.exists(AFB02),
            "AFB03": os.path.exists(AFB03),
            "AFB04": os.path.exists(AFB04),
            "AFB0510": os.path.exists(AFB0510),
            "AFB0610": os.path.exists(AFB0610),
            "AFB0710": os.path.exists(AFB0710),
            "AFB08": os.path.exists(AFB08),
            "AFB8A10": os.path.exists(AFB8A10),
            "AFB0910": os.path.exists(AFB0910),
            "AFB1210": os.path.exists(AFB1210),
        }
        for key, value in file_check.items():
            print(f"{key}: {value}")

        if all(file_check.values()):
            print(print_time() + "Raster_Process :: Start Raster Process Please Wait....")
            Loop_Limit += 1
            t = Timer(3, loadCooldown)
            t.start()
            t.join()

            try:
                print(print_time()+"Raster_Process :: Raster Process " + Full_name[:22])

                threshold_value = 0

                with rasterio.open(AFB08) as src:
                    transform = src.transform
                    crs = src.crs
                    data_AFB08 = src.read(1)
                    print("Shape of data_AFB08:", data_AFB08.shape)  

                with rasterio.open(AFB02) as src_AFB02:
                    data_AFB02 = src_AFB02.read(1)
                    print("Shape of data_AFB02:", data_AFB02.shape) 
                    
                with rasterio.open(AFB03) as src_AFB03:
                    data_AFB03 = src_AFB03.read(1)
                    print("Shape of data_AFB03:", data_AFB03.shape) 

                with rasterio.open(AFB04) as src_AFB04:
                    data_AFB04 = src_AFB04.read(1)
                    print("Shape of data_AFB04:", data_AFB04.shape)

                with rasterio.open(AFB0510) as src_AFB0510:
                    data_AFB0510 = src_AFB0510.read(1)
                    print("Shape of data_AFB05:", data_AFB0510.shape)  
                
                with rasterio.open(AFB0610) as src_AFB0610:
                    data_AFB0610 = src_AFB0610.read(1)
                    print("Shape of data_AFB06:", data_AFB0610.shape) 
                
                with rasterio.open(AFB0710) as src_AFB0710:
                    data_AFB0710 = src_AFB0710.read(1)
                    print("Shape of data_AFB07:", data_AFB0710.shape)  
                
                with rasterio.open(BFB08) as src_BFB08, rasterio.open(AFB8A10) as src_AFB8A10:
                    data_BFB08 = src_BFB08.read(1)
                    data_AFB8A10 = src_AFB8A10.read(1)
                    print("Shape of data_AFB08:", data_AFB08.shape)  
                    print("Shape of data_AFB8A:", data_AFB8A10.shape)  

                with rasterio.open(AFB0910) as src_AFB0910:
                    data_AFB0910 = src_AFB0910.read(1) 
                
                with rasterio.open(BFB1210) as src_BFB1210,rasterio.open(AFB1210) as src_AFB1210:
                    data_BFB1210 = src_BFB1210.read(1)
                    data_AFB1210 = src_AFB1210.read(1)
                    print("Shape of data_BFB12:", data_BFB1210.shape) 
                    print("Shape of data_AFB12:", data_AFB1210.shape)

                bfb08_shape = data_BFB08.shape
                data_BFB08 = np.resize(data_BFB08, bfb08_shape)
                data_BFB1210 = np.resize(data_BFB1210, bfb08_shape)
                data_AFB02 = np.resize(data_AFB02, bfb08_shape)
                data_AFB03 = np.resize(data_AFB03, bfb08_shape)
                data_AFB04 = np.resize(data_AFB04, bfb08_shape)
                data_AFB0510 = np.resize(data_AFB0510, bfb08_shape)
                data_AFB0610 = np.resize(data_AFB0610, bfb08_shape)
                data_AFB0710 = np.resize(data_AFB0710, bfb08_shape)
                data_AFB8A10 = np.resize(data_AFB8A10, bfb08_shape)
                data_AFB0910 = np.resize(data_AFB0910, bfb08_shape)
                data_AFB1210 = np.resize(data_AFB1210, bfb08_shape)

                print("Shape of data_AFB02 (Reshape):", data_AFB02.shape) 
                print("Shape of data_AFB03 (Reshape):", data_AFB03.shape)  
                print("Shape of data_AFB04 (Reshape):", data_AFB04.shape)
                print("Shape of data_AFB05 (Reshape):", data_AFB0510.shape)  
                print("Shape of data_AFB06 (Reshape):", data_AFB0610.shape) 
                print("Shape of data_AFB07 (Reshape):", data_AFB0710.shape)  
                print("Shape of data_AFB08 (Reshape):", data_AFB08.shape)  
                print("Shape of data_AFB8A (Reshape):", data_AFB8A10.shape)  
                print("Shape of data_AFB09 (Reshape):", data_AFB0910.shape)  
                print("Shape of data_BFB12 (Reshape):", data_BFB1210.shape) 
                print("Shape of data_AFB12 (Reshape):", data_AFB1210.shape) 

                # Calculate rNDI and dNDVI
                PreNBR_data = (data_BFB08 - data_BFB1210) / (data_BFB08 + data_BFB1210)
                PostNBR_data = (data_AFB08 - data_AFB1210) / (data_AFB08 + data_AFB1210)
                rNDI = (data_BFB08 - data_AFB08) / (data_BFB08 + data_AFB08)    
                dNBR = PreNBR_data - PostNBR_data
                dNDVI = (data_AFB08 - data_AFB04) / (data_AFB08 + data_AFB04)

                print(print_time() + "Raster_Process :: Burn Raster Condition")

                # Burn condition process
                burnCon_Final = burn_con(rNDI, dNDVI, data_AFB03, data_AFB08, data_AFB8A10, data_AFB02, data_AFB04, data_AFB0510, data_AFB0610, data_AFB0710, data_AFB0910, data_AFB1210)
                
                output_path = os.path.join(Rtbcon, Track, f"{Full_name}.tif")
                save_as_geotiff(burnCon_Final, output_path, transform, crs)



                print(print_time() + "Raster_Process :: RegionGroup")

                # Read the burn condition raster
                with rasterio.open(output_path) as src_BCON:
                    data_BCON = src_BCON.read(1)

                # Burn region process
                threshold_value = 0  # Set your threshold value here
                burnRegion = burn_region(data_BCON, threshold_value)

                output_path = os.path.join(Rtbreg, Track, f"{Short_name}_B12.tif")
                save_as_geotiff(burnRegion, output_path, transform, crs)

                print(print_time() + "Raster_Process :: Burn Raster Process Complete")


                with rasterio.open(BFB08) as src:
                    bounds = src.bounds
                    width, height = src.width, src.height
                    crs = src.crs

                    lats = np.linspace(bounds.top, bounds.bottom, height)
                    longs = np.linspace(bounds.left, bounds.right, width)
                    lon_grid, lat_grid = np.meshgrid(longs, lats)
                    lat_list = lat_grid.ravel()
                    lon_list = lon_grid.ravel()

                    lat_wgs84, lon_wgs84 = transform(crs, 'EPSG:4326', lon_list, lat_list)
                    
                    band_3_data = data_AFB03.ravel()
                    band_4_data = data_AFB04.ravel()
                    band_5_data = data_AFB0510.ravel()
                    band_6_data = data_AFB0610.ravel()
                    band_7_data = data_AFB0710.ravel()
                    band_8_data = data_AFB08.ravel()
                    band_8A_data = data_AFB8A10.ravel()
                    band_9_data = data_AFB0910.ravel()
                    band_12_data = data_AFB1210.ravel()
                    label_data = burnRegion.ravel()

                    df = pd.DataFrame({
                            'Tile': tile,
                            'Date': date,
                            'Latitude_WGS84': lon_wgs84, # Lat and lng wgs84 is flip verticle map so i need to swap value 
                            'Longitude_WGS84': lat_wgs84,
                            'Band_3_Post': band_3_data,
                            'Band_4_Post': band_4_data,
                            'Band_5_Post': band_5_data,
                            'Band_6_Post': band_6_data,
                            'Band_7_Post': band_7_data,
                            'Band_8_Post': band_8_data,
                            'Band_8A_Post': band_8A_data,
                            'Band_9_Post': band_9_data,
                            'Band_12_Post': band_12_data,
                            'Label': label_data
                        })

                    df.fillna(0, inplace=True)

                    output_filename = f"{Full_name[:-6]}.csv" 
                    output_dir = os.path.join(Rtbcon, Track)
                    os.makedirs(output_dir, exist_ok=True)  # Create the subdirectory if it doesn't exist
                    output_path = os.path.join(output_dir, output_filename)
                    df.to_csv(output_path, index=False)

                    # # Copy AFB02, AFB03, and AFB04 images to the output directory
                    # shutil.copy(AFB02, output_dir)
                    # shutil.copy(AFB03, output_dir)
                    # shutil.copy(AFB04, output_dir)

                print(print_time()+"Raster_Process :: Burn Raster Process Complate")

                def reclassify_raster(input_raster, output_raster, remap_dict):
                    with rasterio.open(input_raster) as src:
                        data = src.read(1)
                        profile = src.profile

                        for old_value, new_value in remap_dict.items():
                            data = np.where(data == old_value, new_value, data)

                        output_dir = os.path.dirname(output_raster)
                        os.makedirs(output_dir, exist_ok=True)  # Ensure the directory exists

                        with rasterio.open(output_raster, 'w', **profile) as dst:
                            dst.write(data, 1)

                burnRegion = os.path.join(Rtbreg, Track, f"{Short_name}_B12.tif")
                burnReclass = os.path.join(Rtbreg, Track, f"{Short_name}_BurnReclass.tif")
                remap_dict = {1: 1}

                reclassify_raster(burnRegion, burnReclass, remap_dict)

                print("After burn reclasss")

                def raster_to_polygon(input_raster, output_shapefile, simplify=0, value_field="VALUE"):
                    with rasterio.open(input_raster) as src:
                        image = src.read(1)
                        shapes = rasterio.features.shapes(image, transform=src.transform)

                        geometries = []
                        values = []

                        for geom, value in shapes:
                            if simplify > 0:
                                geom = shape(geom).simplify(simplify)

                            geometries.append(geom)
                            values.append(value)

                        gdf = gpd.GeoDataFrame({value_field: values, 'geometry': geometries})

                        if simplify > 0:
                            gdf['geometry'] = gdf['geometry'].apply(lambda x: x.simplify(simplify))

                        dissolved_gdf = gdf.dissolve(by=value_field)
                        os.makedirs(os.path.dirname(output_shapefile), exist_ok=True)  # Ensure directory exists
                        dissolved_gdf.to_file(output_shapefile)

                burnReclass = os.path.join(Rtbreg, Track, f"{Short_name}_BurnReclass.tif")
                output_shapefile = os.path.join(RtbShape, Track, f"{Short_name}_B12.shp")
                simplify_value = 0.1

                raster_to_polygon(burnReclass, output_shapefile, simplify=simplify_value)
                
                Error_Limit = 2
                print(print_time()+"Plot Image and Label \n \n")


            except Exception as e:
                print(print_time()+"Raster_Process :: !!!!!!!!!! RASTER ERROR !!!!!!!!!!")
                print(print_time() + str(e))
                Error_Limit = Error_Limit - 1
                if Error_Limit < 1 :
                    print(print_time()+"Raster_Process :: !!!!!!!!!! RASTER ERROR 2 Time MoveFile to Image_Missin")
                    Move_File(Mid_name + "B03.jp2", Image + Track, Image_Missing + Track)
                    Move_File(Mid_name + "B04.jp2", Image + Track, Image_Missing + Track)
                    Move_File(Mid_name + "B08.jp2", Image + Track, Image_Missing + Track)
                    Move_File(Mid_name + "B12.jp2", Image + Track, Image_Missing + Track)
                    Move_File(Mid_name + "B1210.jp2", Image + Track, Image_Missing + Track)
                    Move_File(Mid_name + "B1210.jp2.aux.xml", Image + Track, Image_Missing + Track)
                    Move_File(Mid_name + "B1210.jp2.ovr", Image + Track, Image_Missing + Track)
                    Move_File(Mid_name + "B1210.jp2.xml", Image + Track, Image_Missing + Track)
                    Error_Limit = 2

        else:
            print(print_time() + f"Raster_Process :: {Full_name} Image not Found !!!!")
            for key, value in file_check.items():
                print(f"{key}: {value}")
        
    print(print_time() + "Wait New Raster ::")
    mode = True

def main():
    current_dir = os.getcwd()
    print("Current Working Directory:", current_dir)
    global mode, Image_Post
    print(print_time() + "Application Start ::")

    # Get the list of directories inside the 'Image' directory
    image_directories = [d for d in os.listdir(Image) if os.path.isdir(os.path.join(Image, d))]

    for area_dir in image_directories:
        area_path = os.path.join(Image, area_dir)
        rasters = [r for r in os.listdir(area_path) if r.endswith('B1210.jp2')]
        if rasters:
            mode = False
            print(print_time() + f"Found NEW Raster in Area: {area_dir}")
            try:
                Raster_Process(area_dir)
            except Exception as e:
                print(print_time() + "!!!!!!!!!SYSTEM ERROR !!!!!!!!!!!")
                print(print_time() + str(e))
                print(print_time() + "Wait New Raster ::")
        else:
            print(f"No rasters found in directory: {area_path}")

    print(print_time() + f"{len(image_directories)} Area(s) processed.")

main()